# V2.1-P: Importação do pacote congelado (passo 6)

Este notebook é um auxiliar de validação, somente texto. Ele lê o resultado
agregado do congelamento em `temp/v2/v2_package.json` e seu manifesto público
em `config/v2_results.json`. Não há gráficos, dataframes ou modelagem.

O passo 6, regido por `docs/ADR-013-v2-frozen-package.md`, congela o pacote
hierárquico V2 completo: detector crítico, limiar calibrado, referência ao S7
congelado e regra de combinação. Logs brutos e diretórios transitórios do
Kaggle não fazem parte da publicação.


## Por que o portão de reprodução existe

Nenhum modelo ajustado foi persistido no V2.1-D1: o benchmark
publicou apenas agregados e descartou os trinta estimadores junto com
a sessão. Congelar o pacote exige, portanto, reajustar o candidato
selecionado, e reajustar cria a possibilidade de o artefato congelado
diferir daquele que o D1 de fato mediu.

Por isso o congelamento é condicionado a um portão de reprodução
exata, sem nenhuma tolerância numérica. Sob o mesmo caminho de código
do D1, o reajuste precisa reproduzir o limiar calibrado
(-0,13949530151425016), as duas matrizes de confusão, as contagens de
decisões positivas e de sobrescritas efetivas nas duas janelas (57 e
16 na calibração, 258 e 82 na janela externa) e as contagens do pool
de negativos difíceis (946 positivos e 14.190 negativos difíceis).
Cada comparação é registrada como um booleano nomeado.

A comparação é exata nas verificações agregadas. Ela comprova reprodução comportamental nas medidas registradas, mas não prova identidade linha a linha do pool, pois o D1 não persistiu uma assinatura equivalente.

Há exatamente dois desfechos:

- `PACKAGE_FROZEN`: todas as verificações passaram e o bundle joblib
  foi escrito em `artifacts/v2/consumer_complaint_detector_v2.joblib`;
- `REPRODUCTION_MISMATCH`: ao menos uma verificação falhou, NENHUM
  bundle foi escrito e a divergência é publicada como evidência.

Uma divergência não é um contratempo operacional a ser contornado com
novas tentativas. Nada é congelado em silêncio sobre números que não
batem, e a decisão sobre como proceder volta ao Cientista de Dados. O
relatório abaixo diz qual dos dois desfechos ocorreu antes de imprimir
qualquer outra coisa.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v2_import import render_package_import_report

In [2]:
PACKAGE_RESULT_PATH = PROJECT_ROOT / 'temp' / 'v2' / 'v2_package.json'
PACKAGE_MANIFEST_PATH = PROJECT_ROOT / 'config' / 'v2_results.json'


In [3]:
print(render_package_import_report(
    PACKAGE_RESULT_PATH,
    manifest_path=PACKAGE_MANIFEST_PATH,
))


=== V2.1-P FROZEN PACKAGE RESULT ===
V2.1-P FROZEN HIERARCHICAL PACKAGE

OUTCOME
  outcome: PACKAGE_FROZEN
  frozen: True
  complete: True
  status: COMPLETE
  run_mode: full
  runtime_seconds: 1309.026400
  deployment:
    deployment_authorized: False
    status: FROZEN_FOR_CONFIRMATION
    next_step: open_stress_2025_h2_once_under_a_new_confirmatory_protocol

  PACKAGE FROZEN: every reproduction check passed and the
  fitted joblib bundle was written.

REPRODUCTION GATE (EXACT, NO TOLERANCE)
  passed: PASS
  check_count: 21
  comparison: exact_no_tolerance
  source_of_truth: temp/v2/v2_classical_benchmark.json
  candidate_id: word_char_tfidf_union_40000_60000_c_1_hard_negative
  checks:
     check                                            verdict
  -  -----------------------------------------------  -------
  *  calibrated_threshold                             PASS   
  *  calibration_confusion_matrix                     PASS   
  *  calibration_override_decisions                   

## Leitura do relatório

O relatório é somente texto e cada seção `=== ... ===` cobre uma
entrada. Dentro da seção de resultado, OUTCOME vem primeiro e diz por
extenso se um bundle foi persistido. REPRODUCTION GATE lista cada
verificação como PASS ou FAIL, marcando com `*` as verificações
canônicas nomeadas em `required_checks`, e imprime um bloco
DIVERGENCES sempre que alguma verificação falha.

Uma divergência de matriz de confusão aparece apenas como resumo
agregado da diferença: células divergentes, diferença absoluta total e
máxima e os dois totais. As matrizes em si nunca são impressas, e
tampouco narrativas, identificadores ou índices de linha.

SAFETY MARGIN repete o que a margem não diz. As folgas foram medidas
na mesma janela externa que serviu de superfície de seleção, primeiro
entre os candidatos elegíveis e depois no desafio do D2, de modo que
são folgas de desenvolvimento com viés otimista e não evidência
independente de desempenho futuro. A única evidência independente virá
do passo 7, que este notebook não autoriza e não executa.

INTEGRITY encerra com a fronteira declarada. Observe que
`persists_fitted_weights` é deliberadamente verdadeiro aqui, porque
congelar um pacote é precisamente persistir pesos ajustados, enquanto
`persists_narratives_or_identifiers` permanece falso.

Este notebook não decide nada sozinho; ele apenas torna a evidência
publicada legível para revisão humana.